# 04 - Análise de Motivos e Perfil de GC em Sequências Longas

Este notebook reproduz as análises de interpretabilidade e perfil de GC sobre contigs longos (`long/long_loss_function.ipynb`),
fatiando contigs genômicos em janelas de 512 bp e mapeando ativações de motivos e teor de GC via `viral_cnn.interpretability.motifs`.

**Invariantes científicas (`AD-004`, `AD-014`):**
- Rastreamento de ativações de 7-mers ao longo de contigs genômicos extensos.
- Correlação com conteúdo GC local.


In [1]:
import os
import sys
import json
from pathlib import Path

# Resolução autônoma de caminho para JupyterHub remoto e desenvolvimento local
current_dir = Path.cwd().resolve()
for r in [current_dir, current_dir.parent, current_dir.parent.parent, current_dir.parent.parent.parent]:
    if (r / "src" / "viral_cnn").exists():
        if str(r / "src") not in sys.path:
            sys.path.insert(0, str(r / "src"))
        break

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from viral_cnn.notebook_runtime import setup_notebook_runtime
from viral_cnn.inference.voter import slice_sequence
from viral_cnn.interpretability.motifs import scan_motif_occurrences

# 1. Inicialização do runtime e contexto de execução
ctx = setup_notebook_runtime()
output_dir = Path("results_reproduction/long_voter")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Device: {ctx.device}")
print(f"Smoke Test Mode: {ctx.is_smoke_test}")
print(f"Data Root: {ctx.data_root}")
print(f"Output Dir: {output_dir}")


/home/mlima/workspace/envs/rapids-new/lib/python3.12/site-packages/torch/__init__.py:1624: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


Device: cuda
Smoke Test Mode: False
Data Root: /home/mlima/workspace
Output Dir: results_reproduction/long_voter


## 2. Fatiamento em Janelas e Mapeamento de GC e Motivos de 7-mers

Fatia contigs longos em janelas de 512 bp (stride 256 bp) e computa o teor de GC e a ocorrência de motivos aprendidos.


In [2]:
# Contig genômico representativo (3.000 bp)
rng = np.random.RandomState(42)
bases = ["A", "C", "G", "T"]
sample_contig = "".join(rng.choice(bases, size=3000))

# Fatiamento em janelas de 512 bp com stride 256 bp
windows = slice_sequence(sample_contig, window_size=512, stride=256, handle_short="pad")
token_to_base = ["A", "C", "G", "T", "N"]

motifs_tracked = ["CCCCCCC", "GGGGGGG", "ATATATA", "TTTTTTT"]
window_records = []

for idx, win_tokens in enumerate(windows):
    win = "".join(token_to_base[t] for t in win_tokens)
    gc_pct = 100.0 * sum(b in "GC" for b in win) / len(win)
    rec = {
        "window_idx": idx,
        "start_pos": idx * 256,
        "end_pos": idx * 256 + len(win),
        "gc_percent": round(gc_pct, 2),
    }
    for m in motifs_tracked:
        rec[f"motif_{m}"] = win.count(m)
    window_records.append(rec)

df_motifs = pd.DataFrame(window_records)
summary_csv = output_dir / "long_contig_motif_summary.csv"
df_motifs.to_csv(summary_csv, index=False)

print("Resumo de motivos e GC por janela de 512 bp:")
print(df_motifs.to_string(index=False))


Resumo de motivos e GC por janela de 512 bp:
 window_idx  start_pos  end_pos  gc_percent  motif_CCCCCCC  motif_GGGGGGG  motif_ATATATA  motif_TTTTTTT
          0          0      512       45.90              0              0              0              0
          1        256      768       46.68              0              0              0              0
          2        512     1024       46.29              0              0              0              0
          3        768     1280       48.44              0              0              0              0
          4       1024     1536       53.52              0              0              0              0
          5       1280     1792       51.56              0              0              0              0
          6       1536     2048       51.56              0              0              0              0
          7       1792     2304       50.20              0              0              0              0
          8       2

## 3. Geração da Figura de Perfil de GC e Ocorrência de Motivos


In [3]:
fig_path = output_dir / "figure_long_motifs_gc.png"
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

positions = df_motifs["start_pos"]
ax1.plot(positions, df_motifs["gc_percent"], color="teal", marker="o", linewidth=2)
ax1.set_ylabel("Teor de GC (%)")
ax1.set_title("Perfil de GC e Ativação de Motivos ao Longo de Contig Viral Longo (AD-010, AD-014)")
ax1.grid(True, linestyle="--", alpha=0.5)

ax2.bar(positions - 30, df_motifs["motif_CCCCCCC"], width=60, label="CCCCCCC", color="darkblue", alpha=0.7)
ax2.bar(positions + 30, df_motifs["motif_GGGGGGG"], width=60, label="GGGGGGG", color="darkgreen", alpha=0.7)
ax2.set_xlabel("Posição no Contig Genômico (bp)")
ax2.set_ylabel("Contagem de Motivos")
ax2.legend()
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig(fig_path, dpi=300)
plt.close(fig)

print(f"Figura gerada com sucesso: {fig_path}")


Figura gerada com sucesso: results_reproduction/long_voter/figure_long_motifs_gc.png


## 4. Emissão do Manifesto de Motivos em Contigs Longos


In [4]:
manifest_04 = {
    "notebook": "04_long_motifs_and_gc_analysis.ipynb",
    "contig_length": len(sample_contig),
    "n_windows": len(windows),
    "motifs_tracked": motifs_tracked,
    "artifacts": {
        "summary_csv": str(summary_csv),
        "figure": str(fig_path),
    },
}

manifest_path = output_dir / "manifest_04_long_motifs.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest_04, f, indent=2)

print(f"Manifesto 04 salvo em: {manifest_path}")


Manifesto 04 salvo em: results_reproduction/long_voter/manifest_04_long_motifs.json
